# 01 — Exploratory Data Analysis (ISIC-2018)

Inspect image/mask pairs, sizes, and lesion-area distribution before training.
Run `python -m src.download_data --source kaggle` (or `--source synthetic`) first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import load_config, resolve
cfg = load_config('../configs/config.yaml')
df = pd.read_csv(resolve(cfg['data']['splits_dir']) / 'train.csv')
print('train pairs:', len(df))
df.head()

In [ ]:
# Visualise a few image/mask pairs
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for ax_row, (_, row) in zip(axes, df.sample(3, random_state=0).iterrows()):
    img = cv2.cvtColor(cv2.imread(str(resolve(row['image_path']))), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(resolve(row['mask_path'])), cv2.IMREAD_GRAYSCALE)
    ax_row[0].imshow(img); ax_row[0].set_title('image'); ax_row[0].axis('off')
    ax_row[1].imshow(mask, cmap='gray'); ax_row[1].set_title('mask'); ax_row[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Lesion-area fraction distribution (class imbalance check)
fracs = []
for _, row in df.iterrows():
    m = cv2.imread(str(resolve(row['mask_path'])), cv2.IMREAD_GRAYSCALE)
    fracs.append((m > 127).mean())
fracs = np.array(fracs)
print(f'mean lesion area: {fracs.mean():.3f}  (foreground is the minority class)')
plt.hist(fracs, bins=30); plt.xlabel('lesion area fraction'); plt.ylabel('count')
plt.title('Lesion area distribution'); plt.show()